# Анализ туристической отрасли в России за 2018-2023 года

**Цель проекта:** выявить региональные лидеров туристического рынка и оценить динамику спроса на внутренние и зарубежные туры за 2018–2023 годы. Результаты анализа будут использованы для корректировки маркетинговой стратегии и персонализации рекомендаций пользователям сервиса.

## Описание данных

Таблица `hotel.csv` с числом гостиниц, хостелов, санаторно-курортных организаций и мест в них по субъектам Российской Федерации за 2018–2023 годы:
- `Субъект` — наименование субъекта Российской Федерации.
- `Число гостиниц`.
- `Число мест в гостиницах`.
- `Число хостелов`.
- `Число мест в хостелах`.
- `Число санаторно-курортных организаций`.
- `Число мест в санаторно-курортных организациях`.

Таблица `count_person_hotel.csv` с количеством людей, размещённых в гостиницах по субъектам Российской Федерации в 2023 году:
- `Субъект` — наименование субъекта Российской Федерации.
- `Численность лиц, размещенных в гостиницах в 2023 году`.
- `Численность граждан России, размещенных в гостиницах в 2023 году`.
- `Численность иностранных граждан, размещенных в гостиницах в 2023 году`.

Таблица `tour_firm.csv` с числом реализованных турпакетов в 2018-2023 годы:
- `Субъект` — наименование субъекта Российской Федерации.
- `Общее число турпакетов, реализованных населению`.
- `Общее число турпакетов, реализованных гражданам России по территории России`.
- `Общее число турпакетов, реализованных гражданам России по другим странам`.
- `Общее число турпакетов, реализованных гражданам других стран по территории России`.

Таблица `tour_cost_2023.csv` с основными показателями деятельности туристических фирм по субъектам Российской Федерации в 2023 году:
- `Субъект` — наименование субъекта Российской Федерации.
- `Стоимость реализованных турпакетов в 2023 году, млн руб.`.
- `Стоимость турпакетов  реализованных гражданам России по территории России в 2023 году, млн руб.`.
- `Стоимость турпакетов  реализованных гражданам России по другим странам в 2023 году, млн руб.`.

## Подготовка данных


### 1. Загрузка и первичный обзор данных
Подготовка инструментов. Используем **pandas** для манипуляций с данными. Загружаем исходныe датасеты из `data/raw/`.   
На этом этапе мы оценим структуру таблиц и корректность считывания признаков.

In [1]:
import pandas as pd

In [2]:
df_hotel = pd.read_csv('../data/raw/hotel.csv', sep=';')
df_person_hotel = pd.read_csv('../data/raw/count_person_hotel.csv', sep=';')
df_tour_firm = pd.read_csv('../data/raw/tour_firm.csv', sep=';')
df_tour_cost_2023 = pd.read_csv('../data/raw/tour_cost_2023.csv', sep=';')

In [3]:
datasets = {
    "Гостиницы и хостелы": df_hotel,
    "Численность проживающих": df_person_hotel,
    "Турфирмы": df_tour_firm,
    "Стоимость туров 2023": df_tour_cost_2023
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} строк, {df.shape[1]} столбцов")


Гостиницы и хостелы: 81 строк, 37 столбцов
Численность проживающих: 82 строк, 4 столбцов
Турфирмы: 82 строк, 25 столбцов
Стоимость туров 2023: 87 строк, 4 столбцов


In [4]:
for name, df in datasets.items():
    print(f"📌 {name}:")
    print(df.columns)


📌 Гостиницы и хостелы:
Index(['Субъект', 'Число гостиниц, 2018', 'Число гостиниц, 2019',
       'Число гостиниц, 2020', 'Число гостиниц, 2021', 'Число гостиниц, 2022',
       'Число гостиниц, 2023', 'Число мест в гостиницах, 2018',
       'Число мест в гостиницах, 2019', 'Число мест в гостиницах, 2020',
       'Число мест в гостиницах, 2021', 'Число мест в гостиницах, 2022',
       'Число мест в гостиницах, 2023', 'Число хостелов, 2018',
       'Число хостелов, 2019', 'Число хостелов, 2020', 'Число хостелов, 2021',
       'Число хостелов, 2022', 'Число хостелов, 2023',
       'Число мест в хостелах, 2018', 'Число мест в хостелах, 2019',
       'Число мест в хостелах, 2020', 'Число мест в хостелах, 2021',
       'Число мест в хостелах, 2022', 'Число мест в хостелах, 2023',
       'Число санаторно-курортных организаций, 2018',
       'Число санаторно-курортных организаций, 2019',
       'Число санаторно-курортных организаций, 2020',
       'Число санаторно-курортных организаций, 2021',
 

Со столбцами можно работать и в текущем виде, однако лучше предварительно очистить их названия: убрать лишние пробелы, спецсимволы и заменить длинные фразы на короткие, понятные и удобные имена и привести к `snake_case`

In [5]:
df_hotel.columns = (
    df_hotel.columns.str.strip().str.lower()
    .str.replace('субъект', 'region', regex=False)
    .str.replace('число ', '', regex=False)
    .str.replace('мест в ', 'places_in_', regex=False)
    .str.replace(r'гостиниц, |гостиницах, ', 'hotel_', regex=True)
    .str.replace(r'хостелов, |хостелах, ', 'hostel_', regex=True)
    .str.replace(r'санаторно-курортных организаций, |санаторно-курортных организациях, ', 'sanatorium_', regex=True)
    .str.replace(' ', '_', regex=False)
)


In [6]:
df_hotel.columns

Index(['region', 'hotel_2018', 'hotel_2019', 'hotel_2020', 'hotel_2021',
       'hotel_2022', 'hotel_2023', 'places_in_hotel_2018',
       'places_in_hotel_2019', 'places_in_hotel_2020', 'places_in_hotel_2021',
       'places_in_hotel_2022', 'places_in_hotel_2023', 'hostel_2018',
       'hostel_2019', 'hostel_2020', 'hostel_2021', 'hostel_2022',
       'hostel_2023', 'places_in_hostel_2018', 'places_in_hostel_2019',
       'places_in_hostel_2020', 'places_in_hostel_2021',
       'places_in_hostel_2022', 'places_in_hostel_2023', 'sanatorium_2018',
       'sanatorium_2019', 'sanatorium_2020', 'sanatorium_2021',
       'sanatorium_2022', 'sanatorium_2023', 'places_in_sanatorium_2018',
       'places_in_sanatorium_2019', 'places_in_sanatorium_2020',
       'places_in_sanatorium_2021', 'places_in_sanatorium_2022',
       'places_in_sanatorium_2023'],
      dtype='object')

In [7]:
df_person_hotel.columns = [
    "region",
    "total_guests_2023",
    "russian_guests_2023",
    "foreign_guests_2023"
]

In [8]:
df_person_hotel.columns

Index(['region', 'total_guests_2023', 'russian_guests_2023',
       'foreign_guests_2023'],
      dtype='object')

In [9]:
df_tour_firm.columns = (
    df_tour_firm.columns.str.strip()
    .str.replace('Субъект', 'region', case=False)
    .str.replace('Общее число турпакетов, ', 'packages_total_', case=False)
    .str.replace('реализованных населению, ', '', case=False)
    .str.replace('реализованных', '', case=False)
    .str.replace('гражданам России по территории России', 'russian', case=False)
    .str.replace('гражданам России по другим странам', 'foreign', case=False)
    .str.replace('гражданам других стран по территории России', 'foreign_citizens', case=False)
    .str.replace(',', '', regex=False)
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('__', '_', regex=False) 
)


In [10]:
df_tour_firm.columns

Index(['region', 'packages_total_2018', 'packages_total_russian_2018',
       'packages_total_foreign_2018', 'packages_total_foreign_citizens_2018',
       'packages_total_2019', 'packages_total_russian_2019',
       'packages_total_foreign_2019', 'packages_total_foreign_citizens_2019',
       'packages_total_2020', 'packages_total_russian_2020',
       'packages_total_foreign_2020', 'packages_total_foreign_citizens_2020',
       'packages_total_2021', 'packages_total_russian_2021',
       'packages_total_foreign_2021', 'packages_total_foreign_citizens_2021',
       'packages_total_2022', 'packages_total_russian_2022',
       'packages_total_foreign_2022', 'packages_total_foreign_citizens_2022',
       'packages_total_2023', 'packages_total_russian_2023',
       'packages_total_foreign_2023', 'packages_total_foreign_citizens_2023'],
      dtype='object')

In [11]:
df_tour_cost_2023.columns = ['region', 'packages_cost_total_2023', 'packages_cost_ru_domestic_2023', 'packages_cost_ru_abroad_2023']
df_tour_cost_2023.columns

Index(['region', 'packages_cost_total_2023', 'packages_cost_ru_domestic_2023',
       'packages_cost_ru_abroad_2023'],
      dtype='object')

Наименование столбцов были приведены к формату `snake_case` для удобства работы с данными. 

### 2. Очистка данных
Проверим данные на наличие пропусков, дубликатов и аномалий.
- **Типы данных**: убедимся, что все столбцы имеют правильные типы данных (например, числовые столбцы должны быть в формате `int` или `float`, а текстовые — в формате `object`).
- **Пропуски**: выявим и оценим количество пропущенных значений в каждом столбце. Решим, как с ними поступить: удалить строки, заполнить средними/медианными значениями или использовать другие методы.
- **Дубликаты**: проверим наличие дубликатов в данных и удалим их при необходимости.
- **Аномалии**: выявим и проанализируем аномальные значения, которые могут указывать на ошибки в данных или на реальные, но необычные случаи.

In [12]:
for name, df in datasets.items():
    print(f"📌 {name}")
    display(df.dtypes)

📌 Гостиницы и хостелы


region                       object
hotel_2018                    int64
hotel_2019                    int64
hotel_2020                    int64
hotel_2021                    int64
hotel_2022                    int64
hotel_2023                    int64
places_in_hotel_2018          int64
places_in_hotel_2019          int64
places_in_hotel_2020          int64
places_in_hotel_2021          int64
places_in_hotel_2022          int64
places_in_hotel_2023          int64
hostel_2018                   int64
hostel_2019                   int64
hostel_2020                   int64
hostel_2021                   int64
hostel_2022                   int64
hostel_2023                   int64
places_in_hostel_2018        object
places_in_hostel_2019        object
places_in_hostel_2020        object
places_in_hostel_2021        object
places_in_hostel_2022        object
places_in_hostel_2023        object
sanatorium_2018               int64
sanatorium_2019               int64
sanatorium_2020             

📌 Численность проживающих


region                 object
total_guests_2023       int64
russian_guests_2023     int64
foreign_guests_2023     int64
dtype: object

📌 Турфирмы


region                                   object
packages_total_2018                      object
packages_total_russian_2018              object
packages_total_foreign_2018              object
packages_total_foreign_citizens_2018    float64
packages_total_2019                      object
packages_total_russian_2019              object
packages_total_foreign_2019              object
packages_total_foreign_citizens_2019     object
packages_total_2020                      object
packages_total_russian_2020              object
packages_total_foreign_2020              object
packages_total_foreign_citizens_2020    float64
packages_total_2021                      object
packages_total_russian_2021              object
packages_total_foreign_2021             float64
packages_total_foreign_citizens_2021     object
packages_total_2022                      object
packages_total_russian_2022              object
packages_total_foreign_2022              object
packages_total_foreign_citizens_2022    

📌 Стоимость туров 2023


region                            object
packages_cost_total_2023          object
packages_cost_ru_domestic_2023    object
packages_cost_ru_abroad_2023      object
dtype: object

Поля `places_in_hostel_2018`, `places_in_hostel_2019`, `places_in_hostel_2020`, `places_in_hostel_2021`, `places_in_hostel_2022`, `places_in_hostel_2023`, `places_in_sanatorium_2018`, `places_in_sanatorium_2019`, `places_in_sanatorium_2020`, `places_in_sanatorium_2021`, `places_in_sanatorium_2022`, `places_in_sanatorium_2023` в датасете **hotel** имеют тип данных `object`, что может указывать на наличие текстовых значений или пропусков, которые не были корректно обработаны. Необходимо провести дополнительную очистку этих столбцов, чтобы привести их к числовому формату и обеспечить корректный анализ данных.
В датасете `tour_firm.csv` почти все поля, кроме `subject`, имеют тип данных `object`, что может указывать на наличие текстовых значений или пропусков, которые не были корректно обработаны. Необходимо провести дополнительную очистку этих столбцов, чтобы привести их к числовому формату и обеспечить корректный анализ данных.
Датасет `tour_cost_2023.csv` также содержит поля, которые имеют тип данных `object`, что может указывать на наличие текстовых значений или пропусков. Необходимо провести дополнительную очистку этих столбцов, чтобы привести их к числовому формату и обеспечить корректный анализ данных.

In [13]:
def dt_types(df, columns_list):
    for col in columns_list:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    return df 

In [20]:
columns_to_convert = [
    'places_in_hostel_2018', 'places_in_hostel_2019', 'places_in_hostel_2020', 
    'places_in_hostel_2021', 'places_in_hostel_2022', 'places_in_hostel_2023',
    'places_in_sanatorium_2018', 'places_in_sanatorium_2019', 'places_in_sanatorium_2020', 
    'places_in_sanatorium_2021', 'places_in_sanatorium_2022', 'places_in_sanatorium_2023'
]
dt_types(df_hotel, columns_to_convert)
missing_values = df_hotel.isna().sum()
df_hotel.dtypes

region                       object
hotel_2018                    int64
hotel_2019                    int64
hotel_2020                    int64
hotel_2021                    int64
hotel_2022                    int64
hotel_2023                    int64
places_in_hotel_2018          int64
places_in_hotel_2019          int64
places_in_hotel_2020          int64
places_in_hotel_2021          int64
places_in_hotel_2022          int64
places_in_hotel_2023          int64
hostel_2018                   int64
hostel_2019                   int64
hostel_2020                   int64
hostel_2021                   int64
hostel_2022                   int64
hostel_2023                   int64
places_in_hostel_2018         Int64
places_in_hostel_2019         Int64
places_in_hostel_2020         Int64
places_in_hostel_2021         Int64
places_in_hostel_2022         Int64
places_in_hostel_2023         Int64
sanatorium_2018               int64
sanatorium_2019               int64
sanatorium_2020             

In [19]:
columns_to_convert = df_tour_firm.columns[1:]
dt_types(df_tour_firm, columns_to_convert)
df_tour_firm.dtypes

region                                  object
packages_total_2018                      Int64
packages_total_russian_2018              Int64
packages_total_foreign_2018              Int64
packages_total_foreign_citizens_2018     Int64
packages_total_2019                      Int64
packages_total_russian_2019              Int64
packages_total_foreign_2019              Int64
packages_total_foreign_citizens_2019     Int64
packages_total_2020                      Int64
packages_total_russian_2020              Int64
packages_total_foreign_2020              Int64
packages_total_foreign_citizens_2020     Int64
packages_total_2021                      Int64
packages_total_russian_2021              Int64
packages_total_foreign_2021              Int64
packages_total_foreign_citizens_2021     Int64
packages_total_2022                      Int64
packages_total_russian_2022              Int64
packages_total_foreign_2022              Int64
packages_total_foreign_citizens_2022     Int64
packages_tota

In [22]:
df_tour_cost_2023.packages_cost_total_2023 = pd.to_numeric(df_tour_cost_2023.packages_cost_total_2023, errors='coerce')
df_tour_cost_2023.packages_cost_ru_domestic_2023 = pd.to_numeric(df_tour_cost_2023.packages_cost_ru_domestic_2023, errors='coerce')
df_tour_cost_2023.packages_cost_ru_abroad_2023 = pd.to_numeric(df_tour_cost_2023.packages_cost_ru_abroad_2023, errors='coerce')
df_tour_cost_2023.dtypes

region                             object
packages_cost_total_2023          float64
packages_cost_ru_domestic_2023    float64
packages_cost_ru_abroad_2023      float64
dtype: object

Все типы данных были приведены к числовому формату, что позволит нам проводить корректный анализ данных.

In [23]:
for df in datasets.items():
    print(f"📌 {df[0]}:")
    missing_values = df[1].isna().sum()
    print(missing_values[missing_values > 0])

📌 Гостиницы и хостелы:
places_in_hostel_2018        13
places_in_hostel_2019        13
places_in_hostel_2020        19
places_in_hostel_2021        12
places_in_hostel_2022         8
places_in_hostel_2023         9
places_in_sanatorium_2018    11
places_in_sanatorium_2019    11
places_in_sanatorium_2020    10
places_in_sanatorium_2021    12
places_in_sanatorium_2022    12
places_in_sanatorium_2023    10
dtype: int64
📌 Численность проживающих:
Series([], dtype: int64)
📌 Турфирмы:
packages_total_2018                      4
packages_total_russian_2018              4
packages_total_foreign_2018              3
packages_total_foreign_citizens_2018    20
packages_total_2019                      4
packages_total_russian_2019              4
packages_total_foreign_2019              4
packages_total_foreign_citizens_2019    23
packages_total_2020                      4
packages_total_russian_2020              4
packages_total_foreign_2020              4
packages_total_foreign_citizens_2020    39


Почти во всех датасетах были обнаружены пропуски. Принято решение заполнить их нулями, так как в данном случае это может означать отсутствие данных или нулевое значение для соответствующего показателя.

In [24]:
df_hotel.to_csv('../data/processed/hotel.csv', index=False)
df_person_hotel.to_csv('../data/processed/count_person_hotel.csv', index=False)
df_tour_firm.to_csv('../data/processed/tour_firm.csv', index=False)
df_tour_cost_2023.to_csv('../data/processed/tour_cost_2023.csv', index=False)

## Итоги этапа предобработки:
**Названия**: Все признаки переименованы в формат `snake_case` на английском языке.   
**Типы данных**: Столбцы с количественными показателями (количество мест, отелей) приведены к числовому типу (`float64`).   
**Пропуски**: Обнаруженные пропуски (`NaN`) сохранены для предотвращения искажения статистики при расчете средних значений.   
**Артефакты**: Чистые данные экспортированы в `data/processed/` для обеспечения воспроизводимости анализа.